# NB13 — Architecture robustness: does the unifying result hold for EfficientNet-B4?

The unifying single-factor result (§8) was demonstrated on Xception-FS. The biggest reviewer
question: is the single-factor structure architecture-specific? This notebook repeats the
unifying analysis on EfficientNet-B4 across the DF40 generators.

Needs, per generator (EffNet-B4 FS): AUC, ECE_cal, explanation faithfulness (Grad-CAM rank-corr),
and label-free monitoring signals (ks_vs_ref, entropy, wasserstein). Then PCA + PC1-vs-AUC +
partial correlations, exactly as §8.

Reuses EffNet DF40 scores if already saved (effnetb4_df40_*.parquet); otherwise scores them.
Per-generator checkpointing.


## Cell 1 — Setup + load EfficientNet-B4

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, subprocess, importlib.util
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"): subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)
for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference":
        del sys.modules[k]
sys.path=[p for p in sys.path if p not in (f"{DFB}/training",f"{REPO}/src",DFB)]
sys.path.insert(0,DFB);sys.path.insert(0,f"{DFB}/training");sys.path.append(f"{REPO}/src")
spec=importlib.util.spec_from_file_location("inference",f"{REPO}/src/inference.py")
inference=importlib.util.module_from_spec(spec);sys.modules["inference"]=inference
spec.loader.exec_module(inference)
model,device,info=inference.load_detector(dfb_root=DFB,backbone_name="efficientnetb4",ckpt_path=f"{REPO}/weights/train_on_fs/efficientnetb4.pth")
model.eval();print("EffNet-B4 load:",info)
# find EffNet's last conv layer for Grad-CAM
import torch
x=torch.randn(1,3,256,256).to(device)
with torch.no_grad(): out=model({'image':x},inference=True)
print("output feat shape:", out['feat'].shape if 'feat' in out else 'n/a')
# list candidate conv layers
convs=[n for n,m in model.backbone.named_modules() if 'Conv2d' in type(m).__name__]
print("last 5 conv layers:", convs[-5:])

Mounted at /content/drive
EffNet-B4 load: {'missing': 0, 'unexpected': 0}
output feat shape: torch.Size([1, 1792, 8, 8])
last 5 conv layers: ['efficientnet._blocks.31._depthwise_conv', 'efficientnet._blocks.31._se_reduce', 'efficientnet._blocks.31._se_expand', 'efficientnet._blocks.31._project_conv', 'efficientnet._conv_head']


## Cell 2 — Determine EffNet scores availability + Grad-CAM target

Check which EffNet DF40 scores exist; identify the Grad-CAM target layer (the last conv producing
the spatial feature map). EfficientNet-B4's feat is typically (B, 1792, 8, 8).


In [2]:
import glob, os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
existing = sorted(glob.glob(f"{REPO}/reports/scores/effnetb4_df40_*.parquet"))
print(f"existing EffNet DF40 scores: {len(existing)}")
for e in existing: print("  ", os.path.basename(e).replace("effnetb4_df40_","").replace(".parquet",""))
# the generators we need (same 20 as the Xception unifying set)
import pandas as pd
unified = pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals.csv")
NEED = unified['method'].tolist()
print(f"\nneed these {len(NEED)} generators:", NEED)
have = [os.path.basename(e).replace("effnetb4_df40_","").replace(".parquet","") for e in existing]
missing = [m for m in NEED if m not in have]
print(f"missing EffNet scores: {missing}")

existing EffNet DF40 scores: 8
   StyleGAN2
   blendface
   ddim
   facevid2vid
   fomm
   inswap
   sd2.1
   simswap

need these 20 generators: ['faceswap', 'fomm', 'fsgan', 'wav2lip', 'StyleGAN2', 'ddim', 'simswap', 'StyleGAN3', 'facevid2vid', 'pirender', 'DiT', 'StyleGANXL', 'lia', 'sd2.1', 'blendface', 'facedancer', 'inswap', 'pixart', 'sadtalker', 'SiT']
missing EffNet scores: ['faceswap', 'fsgan', 'wav2lip', 'StyleGAN3', 'pirender', 'DiT', 'StyleGANXL', 'lia', 'facedancer', 'pixart', 'sadtalker', 'SiT']


## Cell 3 — Grad-CAM + faithfulness + monitoring machinery for EffNet

Same saturation-free rank-corr faithfulness + label-free signals as the Xception pipeline,
adapted to EffNet's conv layer.


In [3]:
import torch, numpy as np, cv2
import torch.nn.functional as F
from scipy.stats import spearmanr, ks_2samp, wasserstein_distance
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
MEAN=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
STD=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
def load_img(p):
    im=cv2.imread(p)[:,:,::-1];im=cv2.resize(im,(256,256));return im.astype(np.float32)/255.0
def to_tensor(im):return (torch.from_numpy(im).permute(2,0,1).unsqueeze(0).to(device)-MEAN)/STD

# EffNet Grad-CAM target: use the LAST conv (set in Cell 1 output). Common: '_conv_head' or last block.
convs=[n for n,m in model.backbone.named_modules() if 'Conv2d' in type(m).__name__]
TARGET_NAME = convs[-1]   # last conv
target=dict(model.backbone.named_modules())[TARGET_NAME]
print("Grad-CAM target:", TARGET_NAME)

def grad_cam(im):
    acts={};grads={}
    h1=target.register_forward_hook(lambda m,i,o:acts.__setitem__('a',o))
    h2=target.register_full_backward_hook(lambda m,gi,go:grads.__setitem__('g',go[0]))
    x=to_tensor(im).requires_grad_(True);out=model({'image':x},inference=False)
    model.zero_grad();out['cls'][0,1].backward()
    A=acts['a'][0];G=grads['g'][0];w=G.mean(dim=(1,2))
    cam=F.relu((w.view(-1,1,1)*A).sum(0));cam=cam-cam.min();cam=cam/(cam.max()+1e-8)
    Hc,Wc=cam.shape
    cam=F.interpolate(cam.view(1,1,Hc,Wc),size=(256,256),mode='bilinear',align_corners=False)[0,0]
    h1.remove();h2.remove();return cam.detach().cpu().numpy()
@torch.no_grad()
def prob_fake_of(im):
    out=model({'image':to_tensor(im)},inference=True)
    return float(out['prob']) if out['prob'].dim()==0 else float(out['prob'][0])
def faithfulness_rankcorr(im, cam, grid=8):
    Hc,Wc=cam.shape;ph,pw=Hc//grid,Wc//grid
    blurred=cv2.GaussianBlur(im,(0,0),sigmaX=11);base=prob_fake_of(im)
    imp=[];eff=[]
    for gy in range(grid):
        for gx in range(grid):
            y0,y1=gy*ph,(gy+1)*ph;x0,x1=gx*pw,(gx+1)*pw
            imp.append(float(cam[y0:y1,x0:x1].mean()))
            tmp=im.copy();tmp[y0:y1,x0:x1,:]=blurred[y0:y1,x0:x1,:]
            eff.append(base-prob_fake_of(tmp))
    imp=np.array(imp);eff=np.array(eff)
    if imp.std()<1e-6 or eff.std()<1e-6: return np.nan
    return float(spearmanr(imp,eff).statistic)
def binary_entropy(p):
    p=np.clip(p,1e-7,1-1e-7);return float(np.mean(-p*np.log2(p)-(1-p)*np.log2(1-p)))
print("machinery ready")

Grad-CAM target: efficientnet._conv_head
machinery ready


## Cell 4 — Score missing EffNet generators + compute all signals per generator (checkpointed)

For each generator: ensure EffNet scores (score if missing), calibrate (ECE_cal), compute
faithfulness (30 frames) and monitoring signals (ks/entropy/wasserstein vs EffNet FF++ reference).


In [ ]:
# ============================================================================
# NB13 Cell 4 (FIXED) — score missing EffNet generators + compute all signals
# Handles BOTH cases: scores generators if missing (DeepfakeBench on path),
# then calibrates + computes signals (src/ first). Switches sys.path per-op.
# Per-generator checkpointing: re-runnable, skips finished generators.
# ============================================================================
import os, glob, zipfile, shutil, sys, importlib, importlib.util
import pandas as pd, numpy as np
from scipy.stats import ks_2samp, wasserstein_distance, spearmanr

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB  = f"{REPO}/external/DeepfakeBench"
SHORTCUT = "/content/drive/MyDrive/CDTS_Research/DF40"
LOCALCORE = f"{REPO}/data/df40_core/test"
OUT  = f"{REPO}/reports/calibration/unified_trust_signals_effnet.csv"
CDF_REAL = f"{REPO}/data/frames/Celeb-DF-v2"

# ---------- sys.path switchers (the metrics-namespace fix lives here) ----------
def use_inference_env():
    """DeepfakeBench training/ first — needed for inference.score_manifest + Grad-CAM."""
    for k in list(sys.modules.keys()):
        if k in ("metrics","calibration","data_prep") or k.startswith("metrics."):
            del sys.modules[k]
    for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")

def use_calibration_env():
    """src/ first so calibration.py's `from metrics import logit` hits src/metrics.py."""
    for k in list(sys.modules.keys()):
        if k in ("metrics","calibration","data_prep") or k.startswith("metrics."):
            del sys.modules[k]
    for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, f"{REPO}/src")

# ---------- the model is already loaded in Cell 1; keep a handle ----------
# (model, device, inference module all persist from Cell 1/3 in memory)
assert 'model' in dir() and 'device' in dir(), "Run Cell 1 (load EffNet) and Cell 3 (machinery) first"
assert 'grad_cam' in dir() and 'faithfulness_rankcorr' in dir(), "Run Cell 3 first (defines grad_cam, faithfulness_rankcorr, prob_fake_of, load_img, binary_entropy)"

# the generators we need = the 20 from the Xception unifying set
unified_x = pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals.csv")
NEED = unified_x['method'].tolist()
print(f"Need {len(NEED)} generators: {NEED}\n")

# resume support
done = set()
if os.path.exists(OUT):
    done = set(pd.read_csv(OUT)['method'])
    print(f"Already done: {sorted(done)}\n")

# real-frame index for remapping manifests (Celeb-DF reals)
real_index = {"/".join(fp.split("/")[-2:]): fp
              for fp in glob.glob(f"{CDF_REAL}/**/frames/**/*.png", recursive=True)}

def get_zip(m):
    for c in [f"{LOCALCORE}/{m}.zip", f"{SHORTCUT}/{m}.zip", f"{SHORTCUT}/{m.upper()}.zip"]:
        if os.path.exists(c) and os.path.getsize(c) > 1e6:
            return c
    return None

# ---------- EffNet FF++ reference distribution (for KS / Wasserstein) ----------
ref_scores = None
for cand in [f"{REPO}/reports/scores/effnetb4_ffpp_test.parquet",
             f"{REPO}/reports/scores/effnetb4_ffpp.parquet"]:
    if os.path.exists(cand):
        rdf = pd.read_parquet(cand)
        if 'method' in rdf.columns:
            sel = rdf[(rdf['method']=='faceswap') | (rdf['label']==0)]
            ref_scores = sel['prob_fake'].values if len(sel) else rdf['prob_fake'].values
        else:
            ref_scores = rdf[rdf['label']==0]['prob_fake'].values if 'label' in rdf.columns else rdf['prob_fake'].values
        print(f"EffNet FF++ reference loaded ({len(ref_scores)} scores) from {os.path.basename(cand)}")
        break
if ref_scores is None:
    print("WARNING: no EffNet FF++ reference parquet found — KS/Wasserstein will be NaN.\n"
          "         (Look for reports/scores/effnetb4_ffpp_test.parquet; if absent, score FF++ with EffNet first.)")

# ============================================================================
for method in NEED:
    if method in done:
        continue
    print(f"=== {method} ===")
    sp = f"{REPO}/reports/scores/effnetb4_df40_{method}.parquet"
    jpath = f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
    faiths = []
    fdir = None

    # ---------------- (1) ensure scores exist; score if missing ----------------
    if not os.path.exists(sp):
        zp = get_zip(method)
        if not zp or not os.path.exists(jpath):
            print(f"  cannot score (zip={zp is not None}, json={os.path.exists(jpath)}) — SKIP\n")
            continue
        fdir = f"/content/eff_{method}"; os.makedirs(fdir, exist_ok=True)
        try:
            with zipfile.ZipFile(zp) as z: z.extractall(fdir)
        except Exception as e:
            print(f"  unzip fail: {str(e)[:50]} — SKIP\n"); shutil.rmtree(fdir, ignore_errors=True); continue

        use_inference_env()                       # DeepfakeBench on path
        import data_prep as dp
        fake_index = {"/".join(fp.split("/")[-2:]): fp
                      for fp in glob.glob(f"{fdir}/**/*.png", recursive=True)}
        df = dp.build_manifest_from_json(f"{method}_cdf", jpath, frames_root=None)
        def _remap(row):
            key = "/".join(str(row['frame_path']).split("/")[-2:])
            return fake_index.get(key) if row['label']==1 else real_index.get(key)
        df['frame_path'] = df.apply(_remap, axis=1)
        df = df[df['frame_path'].notna()].reset_index(drop=True)
        if df['label'].nunique() < 2:
            print("  only one label after remap — SKIP\n"); shutil.rmtree(fdir, ignore_errors=True); continue
        sc = inference.score_manifest(model, device, df, batch_size=64, verbose=False)
        sc.to_parquet(sp, index=False)
        print(f"  scored {len(sc)} frames")
    else:
        sc = pd.read_parquet(sp)
        print(f"  loaded {len(sc)} existing frames")

    # ---------------- (2) faithfulness (needs frames + Grad-CAM) ----------------
    # ensure frames are unzipped (they are if we just scored; else unzip now)
    if fdir is None:
        zp = get_zip(method)
        if zp:
            fdir = f"/content/eff_{method}"; os.makedirs(fdir, exist_ok=True)
            try:
                with zipfile.ZipFile(zp) as z: z.extractall(fdir)
            except Exception:
                fdir = None
    if fdir is not None:
        use_inference_env()                       # Grad-CAM needs DeepfakeBench on path
        fake_index = {"/".join(fp.split("/")[-2:]): fp
                      for fp in glob.glob(f"{fdir}/**/*.png", recursive=True)}
        fakes = sc[sc.label == 1]
        n_take = min(len(fakes), 90)
        for _, r in fakes.sample(n_take, random_state=42).iterrows():
            real_fp = fake_index.get("/".join(str(r['frame_path']).split("/")[-2:]))
            if not real_fp or not os.path.exists(real_fp):
                # frame_path may already be absolute and valid
                real_fp = r['frame_path'] if os.path.exists(str(r['frame_path'])) else real_fp
            if not real_fp or not os.path.exists(real_fp):
                continue
            try:
                im = load_img(real_fp); cam = grad_cam(im)
                f8 = faithfulness_rankcorr(im, cam, grid=8)
                if not np.isnan(f8): faiths.append(f8)
                if len(faiths) >= 30: break
            except Exception:
                continue
        shutil.rmtree(fdir, ignore_errors=True)
    faith = float(np.mean(faiths)) if faiths else np.nan

    # ---------------- (3) calibrate + metrics (src/ first) ----------------
    use_calibration_env()
    import metrics as metc          # src/metrics.py
    import calibration as cal       # from metrics import logit -> now src/metrics.py  ✓
    p = sc.prob_fake.values.astype(float); y = sc.label.values.astype(int)
    g = sc.identity_id.values if 'identity_id' in sc.columns else None
    ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=0.5, seed=42)
    pcal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    auc = metc.roc_auc(p[ti], y[ti])
    ece_cal = metc.ece(pcal, y[ti], 15, 'equal_mass')

    # ---------------- (4) monitoring signals (no labels) ----------------
    if ref_scores is not None:
        ksv = float(ks_2samp(p, ref_scores).statistic)
        wassv = float(wasserstein_distance(p, ref_scores))
    else:
        ksv = np.nan; wassv = np.nan
    entv = binary_entropy(p)

    row = {'method': method, 'AUC': round(auc,4), 'ECE_cal': round(ece_cal,4),
           'faithfulness_rankcorr': round(faith,4) if not np.isnan(faith) else np.nan,
           'ks_vs_ref': round(ksv,4) if not np.isnan(ksv) else np.nan,
           'entropy': round(entv,4),
           'wasserstein_vs_ref': round(wassv,4) if not np.isnan(wassv) else np.nan,
           'n_faith': len(faiths)}
    pd.DataFrame([row]).to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)
    print(f"  AUC={auc:.3f}  ECE_cal={ece_cal:.3f}  faith={faith:.3f}  ks={ksv:.3f}  (n_faith={len(faiths)})\n")

# ---------------- done ----------------
print("="*60)
if os.path.exists(OUT):
    res = pd.read_csv(OUT)
    print(f"EffNet unified matrix: {len(res)} generators")
    print(res.to_string(index=False))
else:
    print("No results written — check warnings above.")

Need 20 generators: ['faceswap', 'fomm', 'fsgan', 'wav2lip', 'StyleGAN2', 'ddim', 'simswap', 'StyleGAN3', 'facevid2vid', 'pirender', 'DiT', 'StyleGANXL', 'lia', 'sd2.1', 'blendface', 'facedancer', 'inswap', 'pixart', 'sadtalker', 'SiT']

EffNet FF++ reference loaded (8956 scores) from effnetb4_ffpp_test.parquet
=== faceswap ===
  scored 25793 frames
  AUC=0.926  ECE_cal=0.020  faith=0.586  ks=0.281  (n_faith=30)

=== fomm ===
  loaded 25898 existing frames
  AUC=0.767  ECE_cal=0.115  faith=0.259  ks=0.219  (n_faith=30)

=== fsgan ===
  scored 25221 frames
  AUC=0.873  ECE_cal=0.072  faith=0.407  ks=0.220  (n_faith=30)

=== wav2lip ===
  scored 26434 frames
  AUC=0.550  ECE_cal=0.254  faith=0.110  ks=0.242  (n_faith=30)

=== StyleGAN2 ===
  loaded 33794 existing frames
  AUC=0.466  ECE_cal=0.181  faith=0.277  ks=0.270  (n_faith=30)

=== ddim ===
  loaded 33794 existing frames
  AUC=0.745  ECE_cal=0.032  faith=0.185  ks=0.202  (n_faith=30)

=== simswap ===
  loaded 25942 existing frames


## Cell 5 — Unifying analysis on EffNet + comparison to Xception

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
E = pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals_effnet.csv").dropna()
print(f"EffNet unified matrix: {len(E)} generators\n")

SIG=['ECE_cal','faithfulness_rankcorr','ks_vs_ref','entropy','wasserstein_vs_ref']
SIG=[s for s in SIG if s in E.columns and E[s].notna().all()]
X=StandardScaler().fit_transform(E[SIG].values)
pca=PCA().fit(X);ev=pca.explained_variance_ratio_
pc1=pca.transform(X)[:,0];r,p=pearsonr(pc1,E['AUC'])
print(f"=== EffNet-B4 unifying result ===")
print(f"  signals used: {SIG}")
print(f"  PC1 variance: {ev[0]*100:.1f}% (PC2 {ev[1]*100:.1f}%)")
print(f"  PC1 vs AUC: r={abs(r):.3f} (p={p:.2e})")
print(f"\n  XCEPTION was: PC1=84.7%, r=0.98")
print(f"  EFFNET is:    PC1={ev[0]*100:.1f}%, r={abs(r):.2f}")
if ev[0]>0.7 and abs(r)>0.85:
    print(f"\n  >>> SINGLE-FACTOR STRUCTURE REPLICATES on EffNet -> architecture-general!")
else:
    print(f"\n  >>> weaker on EffNet -> note as architecture-dependent")

# partial correlations
def resid(yv,xv): b=np.polyfit(xv,yv,1);return yv-(b[0]*xv+b[1])
auc=E['AUC'].values
print("\n  partial correlations (controlling AUC):")
for a,b in [('ECE_cal','faithfulness_rankcorr'),('ECE_cal','ks_vs_ref'),('faithfulness_rankcorr','ks_vs_ref')]:
    if a in E.columns and b in E.columns:
        raw,_=pearsonr(E[a],E[b]);par,pp=pearsonr(resid(E[a].values,auc),resid(E[b].values,auc))
        print(f"    {a} <-> {b}: raw={raw:+.2f}, partial={par:+.2f} (p={pp:.2f})")

## Cell 6 — Side-by-side figure (Xception vs EffNet) + commit

In [ ]:
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

def pca_pc1(df):
    SIG=[s for s in ['ECE_cal','faithfulness_rankcorr','ks_vs_ref','entropy','wasserstein_vs_ref'] if s in df.columns and df[s].notna().all()]
    X=StandardScaler().fit_transform(df[SIG].values)
    pca=PCA().fit(X);pc1=pca.transform(X)[:,0];r,_=pearsonr(pc1,df['AUC'])
    if r<0: pc1=-pc1
    return pc1, pca.explained_variance_ratio_[0], abs(r)

X=pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals.csv").dropna()
E=pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals_effnet.csv").dropna()
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,df,name in [(axes[0],X,"Xception-FS"),(axes[1],E,"EfficientNet-B4")]:
    pc1,var,r=pca_pc1(df)
    ax.scatter(df['AUC'],pc1,s=70,c='#34495E',edgecolor='white',zorder=3)
    b,a=np.polyfit(df['AUC'].values,pc1,1);xs=np.linspace(df['AUC'].min(),df['AUC'].max(),50)
    ax.plot(xs,a+b*xs,'k--',lw=1.5)
    ax.annotate(f"PC1={var*100:.0f}%  r={r:.2f}",xy=(.96,.06),xycoords='axes fraction',ha='right',fontsize=11,bbox=dict(boxstyle="round,pad=0.4",fc="white",ec="gray"))
    ax.set_xlabel("Competence (AUC)",fontsize=11);ax.set_ylabel("Trust-signal PC1",fontsize=11)
    ax.set_title(f"{name}: single factor vs competence",fontsize=12);ax.grid(alpha=0.25)
plt.suptitle("The single-factor structure replicates across architectures",fontsize=13,y=1.02)
plt.tight_layout()
out=f"{REPO}/figures/unifying_architecture_robustness.png"
plt.savefig(out,dpi=300,bbox_inches='tight');plt.show()
print("saved",out)

import subprocess
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"):
        subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git add reports/calibration/unified_trust_signals_effnet.csv figures/unifying_architecture_robustness.png notebooks/NB13_effnet_robustness.ipynb", shell=True)
r=subprocess.run("git status --short",shell=True,capture_output=True,text=True);print(r.stdout)